# Auditoria das previsões do champion

Compara `review_data/` (previsões originais) com `curated_data/` (anotações humanas) usando scores dos sidecars em `production_predictions/` e `COCOeval`. A mesma implementação é usada por `run_curation.sh` antes de decidir se deve disparar o retreinamento.

O campo `Precision` abaixo reproduz a saída anterior do notebook (`evaluator.stats[0]`), portanto é igual ao mAP@0.50:0.95 e não é uma medida independente de precisão.

In [23]:
import sys
sys.path.append("..")

In [24]:
from src.data_automation.curation_metrics import evaluate_curation, RETRAIN_THRESHOLD

## 1. Converter os dados e executar COCOeval

In [25]:
evaluation = evaluate_curation()

creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.04s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.755
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.788
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.760
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.614
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.822
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.724
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.149
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.779
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.827
 Average Recall     (AR) @[ IoU=0.5

## 3. Métricas e decisão do gate

In [26]:
print(f"Execução: {evaluation.run_id}")
print(f"Imagens: {evaluation.image_count}")
print(f"Predições com score: {evaluation.prediction_count}")
print(f"Caixas humanas: {evaluation.ground_truth_count}")
print(f"Caixas degeneradas incluídas: {evaluation.degenerate_prediction_count}")
if evaluation.metrics is None:
    print("Métricas COCO indefinidas: não há caixas humanas.")
    should_retrain = False
else:
    for name, value in evaluation.metrics.as_dict().items():
        print(f"{name}: {value}")
    should_retrain = evaluation.metrics.all_below(RETRAIN_THRESHOLD)
print(f"RETRAIN_REQUIRED={str(should_retrain).lower()}")


Execução: 20261120T195946194497Z
Imagens: 14
Predições com score: 3187
Caixas humanas: 71
Caixas degeneradas incluídas: 9
mAP@0.50:0.95: 0.7546423059485681
mAP@0.50: 0.7876373567412733
mAP@0.75: 0.7595670246705443
Precision: 0.7546423059485681
Recall: 0.8267605633802818
RETRAIN_REQUIRED=true
